# Silver Table Creation and Incremental Load

This notebook is responsible for:
- Loading configuration and logging utilities.
- Creating the Silver curated table for clinical trial studies.
- Creating supporting tables for rejected and duplicate records.
- Reading the latest incremental batch from the Bronze table.
- Filtering and preparing data for Silver processing.

**Purpose:**  
To curate, validate, and incrementally load clinical trial study data from the Bronze layer into the Silver layer, while logging rejected and duplicate records for further analysis.

##### importing the required libraries

In [0]:
%run ./config

In [0]:
# Importing all functions from pyspark.sql.functions for DataFrame operations
from pyspark.sql.functions import *

# Importing Window for window functions in Spark DataFrames
from pyspark.sql.window import Window

# Importing DeltaTable for Delta Lake operations
from delta.tables import DeltaTable

In [0]:
# Import logging dependencies directly
from datetime import datetime
from pyspark.sql import DataFrame

# Silver logging functions
def log_silver_read(df, source_table, operation="Bronze Read"):
    count = df.count() if isinstance(df, DataFrame) else df
    print(f"[SILVER] [{operation}] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Source: {source_table} | Records: {count:,}")
    return count

def log_silver_transformation(df, transformation_name):
    count = df.count() if isinstance(df, DataFrame) else df
    print(f"[SILVER] [{transformation_name}] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Records: {count:,}")
    return count

def log_silver_validation(total, valid, rejected, validation_type="Validation"):
    success_rate = (valid / total * 100) if total > 0 else 0
    print(f"[SILVER] [{validation_type}] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Total: {total:,} | Valid: {valid:,} | Rejected: {rejected:,} | Success: {success_rate:.2f}%")
    return {"total": total, "valid": valid, "rejected": rejected}

def log_silver_deduplication(total_valid, curated, duplicates, dedup_key="Deduplication"):
    duplicate_rate = (duplicates / total_valid * 100) if total_valid > 0 else 0
    print(f"[SILVER] [{dedup_key}] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Total Valid: {total_valid:,} | Curated: {curated:,} | Duplicates: {duplicates:,} | Dup Rate: {duplicate_rate:.2f}%")
    return {"total_valid": total_valid, "curated": curated, "duplicates": duplicates}

def log_silver_merge(df, table_name, write_mode="merge"):
    count = df.count() if isinstance(df, DataFrame) else df
    print(f"[SILVER] [Merge] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Table: {table_name} | Mode: {write_mode} | Records: {count:,}")
    return count

# Bronze logging functions (needed for silver notebook)
def log_bronze_write(df, table_name, write_mode="append"):
    count = df.count() if isinstance(df, DataFrame) else df
    print(f"[BRONZE] [Write] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Table: {table_name} | Mode: {write_mode} | Records: {count:,}")
    return count

# Gold logging functions (needed for analytics)
def log_gold_analytics(metric_name, metric_value, details=None):
    print(f"[GOLD] [Analytics] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Metric: {metric_name} | Value: {metric_value}")
    return metric_value

print("Logging functions loaded successfully")

##### Creating Schema

In [0]:
# Create silver curated table from config
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {cfg['silver_table']}
USING DELTA
LOCATION '{cfg['silver_path']}'
""")

# Log table creation
print(f"[SILVER] [Table Created] {cfg['silver_table']}")

In [0]:
%skip
%sql

-- Create the table for storing rejected clinical trial records if it does not already exist
CREATE TABLE IF NOT EXISTS
clinical_datalakehouse_modernization.silver.clinical_trial_rejected_records
USING DELTA
LOCATION 's3://clinical-trial4/silver/clinical_trial_rejected_records/';

In [0]:
%skip
%sql
-- Create the table for logging duplicate clinical trial records if it does not already exist
CREATE TABLE IF NOT EXISTS
clinical_datalakehouse_modernization.silver.clinical_trial_duplicate_log
USING DELTA
-- Specify the location of the Delta table in S3
LOCATION 's3://clinical-trial4/silver/clinical_trial_duplicate_log/';

##### Read the latest bronze batch of data - Incremental load (incremental read operation)

In [0]:
%skip
# Read Bronze
bronze_df = spark.table(
    "clinical_datalakehouse_modernization.bronze.clinical_trial_studies_bronze"
)

# Get latest processed version from Silver
last_processed_version = spark.sql("""
SELECT COALESCE(MAX(version), -1)
FROM clinical_datalakehouse_modernization.silver.clinical_trial_studies_curated
""").collect()[0][0]

# Process only new versions
bronze_df = bronze_df.filter(
    col("version") > last_processed_version
)

display(bronze_df)

In [0]:
# Read the bronze clinical trial studies table as a Spark DataFrame from config
try:
    bronze_df = spark.table(cfg['bronze_table'])
    
    # Find the latest version number in the bronze table
    latest_version = bronze_df.agg(
        max("version")
    ).collect()[0][0]
    
    # -- Filter the DataFrame to only include records from the latest version
    # bronze_df = bronze_df.filter(
    #     col("version") == latest_version
    # )
    
    # Print the count of records in the latest bronze batch
    print("Bronze Count :", bronze_df.count())
    
    # Log silver read operation
    log_silver_read(bronze_df, cfg['bronze_table'], "Bronze Data Read")
except Exception as e:
    if "DELTA_TABLE_NOT_FOUND" in str(e) or "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print(f"[INFO] Bronze table does not exist yet: {cfg['bronze_table']}")
        print("[INFO] No data to process. Please run the Bronze load notebook first.")
        # Create empty DataFrame to allow notebook to continue without errors
        bronze_df = spark.createDataFrame([], schema="nct_number STRING, version INT")
    else:
        raise


##### Standardize column names

In [0]:
# Copy bronze_df to silver_df for column standardization
silver_df = bronze_df

# Iterate through each column to standardize its name
for column in silver_df.columns:

    # Convert column name to lowercase and replace spaces, hyphens, and slashes with underscores
    new_column = (
        column.lower()
        .replace(" ","_")
        .replace("-","_")
        .replace("/","_")
    )

    # Rename the column in the DataFrame
    silver_df = silver_df.withColumnRenamed(
        column,
        new_column
    )

# Log column standardization transformation
log_silver_transformation(silver_df, "Column Name Standardization")

##### Data Cleaning and Standardization

In [0]:
# Identify all columns in silver_df that have string data type
string_cols = [
    c for c, dtype in silver_df.dtypes if dtype == "string"
]

# For each string column, clean and standardize values
for c in string_cols:

    silver_df = silver_df.withColumn(
        c,
        # If the value is null or empty, set it to 'UNKNOWN'; otherwise, trim and uppercase the value
        when(
            col(c).isNull() |
            (trim(col(c))==""),
            "UNKNOWN"
        ).otherwise(
            upper(trim(col(c)))
        )
    )

# Log data cleaning transformation
log_silver_transformation(silver_df, "String Data Cleaning and Standardization")

In [0]:
# Fix locations data quality issue - replace date patterns with UNKNOWN
# Some records have dates (e.g., "2016-01-06") in the locations column
silver_df = silver_df.withColumn(
    "locations",
    when(
        # Check if locations matches date pattern (YYYY-MM-DD)
        col("locations").rlike("^[0-9]{4}-[0-9]{2}-[0-9]{2}$"),
        "UNKNOWN"  # Replace with UNKNOWN
    ).otherwise(
        col("locations")  # Keep original value
    )
)

# Log data quality fix
log_silver_transformation(silver_df, "Locations Data Quality Fix - Date Pattern Removal")

In [0]:
# Replace null enrollment values with 0 to indicate missing or unknown enrollment data
silver_df = silver_df.fillna(
    {"enrollment": 0}
)

# Log null handling
log_silver_transformation(silver_df, "Enrollment Null Handling")

##### Pattern validation

In [0]:
# Filter records with valid NCT numbers (must start with 'NCT' followed by 8 digits)
valid_df = silver_df.filter(
    col("nct_number")
    .rlike("^NCT[0-9]{8}$")
)

# Filter records with invalid NCT numbers and add a rejection reason column
rejected_df = silver_df.filter(
    ~col("nct_number")
    .rlike("^NCT[0-9]{8}$")
).withColumn(
    "rejection_reason",
    lit("INVALID NCT FORMAT")
)

# Log validation results
log_silver_validation(silver_df.count(), valid_df.count(), rejected_df.count(), "NCT Number Pattern Validation")

##### Additional Transformtions

In [0]:
# Calculate the study duration in days between completion_date and start_date
valid_df = valid_df.withColumn(
    "study_duration_days",
    datediff(
        col("completion_date"),
        col("start_date")
    )
# Categorize enrollment size into SMALL, MEDIUM, or LARGE buckets
).withColumn(
    "enrollment_bucket",
    when(col("enrollment") < 100,"SMALL")
    .when(col("enrollment") < 1000,"MEDIUM")
    .otherwise("LARGE")
# Flag if the study includes children based on the age column
).withColumn(
    "has_child",
    when(
        col("age").contains("CHILD"),
        1
    ).otherwise(0)
# Flag if the study includes adults based on the age column
).withColumn(
    "has_adult",
    when(
        col("age").contains("ADULT"),
        1
    ).otherwise(0)
# Flag if the study includes older adults based on the age column
).withColumn(
    "has_older_adult",
    when(
        col("age").contains("OLDER_ADULT"),
        1
    ).otherwise(0)
# Extract the year from the first_posted date (handle NULLs with 9999 for 'Unknown')
).withColumn(
    "first_posted_year",
    coalesce(year(col("first_posted")), lit(9999))
# Extract the month from the first_posted date (handle NULLs with 0 for 'Unknown')
).withColumn(
    "first_posted_month",
    coalesce(month(col("first_posted")), lit(0))
)

In [0]:
# Verify the NULL handling for first_posted dates
print("="*60)
print("DATE NULL HANDLING VERIFICATION")
print("="*60)

# Check records where first_posted is NULL
null_date_count = valid_df.filter(col("first_posted").isNull()).count()
print(f"\nRecords with NULL first_posted: {null_date_count:,}")

# Show sample of records with default year/month values
print("\nSample records with NULL dates (will have year=9999, month=0):\n")
display(
    valid_df.filter(col("first_posted").isNull())
    .select("nct_number", "first_posted", "first_posted_year", "first_posted_month")
    .limit(5)
)

print("\n NULL dates are now handled:")
print("   - first_posted_year = 9999 (Unknown Year)")
print("   - first_posted_month = 0 (Unknown Month)")
print("="*60)

In [0]:
# Replace null study_duration_days with 0 to indicate missing or unknown duration
valid_df = valid_df.fillna(
    {"study_duration_days": 0}
)

# Log null handling
log_silver_transformation(valid_df, "Study Duration Null Handling")

In [0]:
# Add a new column 'status_category' to classify study status into broader categories
valid_df = valid_df.withColumn(
    "status_category",
    # If study_status is one of the active recruitment statuses, set category to 'ACTIVE'
    when(
        col("study_status").isin(
            "RECRUITING",
            "NOT_YET_RECRUITING",
            "ENROLLING_BY_INVITATION"
        ),
        "ACTIVE"
    )
    # If study_status is 'COMPLETED', set category to 'COMPLETED'
    .when(
        col("study_status") == "COMPLETED",
        "COMPLETED"
    )
    # If study_status is one of the stopped statuses, set category to 'STOPPED'
    .when(
        col("study_status").isin(
            "TERMINATED",
            "WITHDRAWN",
            "SUSPENDED"
        ),
        "STOPPED"
    )
    # For any other status, set category to 'OTHER'
    .otherwise("OTHER")
)

In [0]:
# Display the distinct combinations of study_status and its broader status_category
display(
    valid_df.select(
        "study_status",      # Original study status from the dataset
        "status_category"    # Categorized status (ACTIVE, COMPLETED, STOPPED, OTHER)
    ).distinct()            # Show only unique pairs
)

##### De-Duplication

In [0]:
# Define a window specification to group by NCT number and order by ingestion timestamp (latest first)
window_spec = Window.partitionBy(
    "nct_number"
).orderBy(
    desc("ingestion_timestamp")
)

# Assign a row number to each record within each NCT group, with the latest record getting row_num = 1
ranked_df = valid_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

# Keep only the latest record for each NCT number (row_num == 1) for the curated silver table
silver_curated_df = ranked_df.filter(
    col("row_num")==1
).drop("row_num")

# Identify and log older duplicate records (row_num > 1) for auditing purposes
duplicate_log_df = ranked_df.filter(
    col("row_num")>1
).drop("row_num") \
.withColumn(
    "duplicate_reason",
    lit("OLDER DUPLICATE RECORD")
)

# Log deduplication results
log_silver_deduplication(valid_df.count(), silver_curated_df.count(), duplicate_log_df.count(), "NCT Number Deduplication")



##### Audit columns

In [0]:
# Validate funder_type, study_status, and phases in silver layer

# List of valid funder_type values
valid_funder_types = [
    "OTHER", "INDUSTRY", "NIH", "FED", "NETWORK", "OTHER_GOV", "INDIV", "UNKNOWN"
]

silver_curated_df = silver_curated_df.withColumn(
    "funder_type",
    when(
        col("funder_type").isNull() | ~col("funder_type").isin(valid_funder_types),
        "UNKNOWN"
    ).otherwise(col("funder_type"))
)

# List of valid study_status values
valid_statuses = [
    "COMPLETED", "RECRUITING", "NOT_YET_RECRUITING", "ACTIVE_NOT_RECRUITING",
    "TERMINATED", "WITHDRAWN", "ENROLLING_BY_INVITATION", "SUSPENDED",
    "WITHHELD", "NO_LONGER_AVAILABLE", "AVAILABLE", "APPROVED_FOR_MARKETING",
    "TEMPORARILY_NOT_AVAILABLE", "UNKNOWN"
]

silver_curated_df = silver_curated_df.withColumn(
    "study_status",
    when(
        col("study_status").isNull() | ~col("study_status").isin(valid_statuses),
        "UNKNOWN"
    ).otherwise(col("study_status"))
)

# List of valid phases values
valid_phases = [
    "PHASE1", "PHASE2", "PHASE3", "PHASE4", "EARLY_PHASE1",
    "PHASE1|PHASE2", "PHASE2|PHASE3", "NA", "UNKNOWN"
]

silver_curated_df = silver_curated_df.withColumn(
    "phases",
    when(
        col("phases").isNull() | ~col("phases").isin(valid_phases),
        "UNKNOWN"
    ).otherwise(col("phases"))
)

print("Validation for funder_type, study_status, and phases applied.")

# Log validation transformation
log_silver_validation(silver_curated_df.count(), silver_curated_df.count(), 0, "Field Value Validation")

In [0]:
# Add a column to record the timestamp when the data is loaded into the silver table
silver_curated_df = silver_curated_df.withColumn(
    "silver_load_timestamp",
    current_timestamp()
).withColumn(
    "batch_id",
    date_format(
        current_timestamp(),
        "yyyyMMddHHmmss"
    )
)

# Log timestamp addition
log_silver_transformation(silver_curated_df, "Silver Load Timestamp and Batch ID")

##### Validation

In [0]:
# bronze_df: All records ingested (bronze = valid + rejected)
print("Bronze :", bronze_df.count())

# valid_df: Records with valid NCT numbers (subset of bronze)
print("Valid :", valid_df.count())

# rejected_df: Records with invalid NCT numbers (subset of bronze)
print("Rejected :", rejected_df.count())

# duplicate_log_df: Older duplicate records within valid_df (duplicates based on NCT number)
print("Duplicates :", duplicate_log_df.count())

# silver_curated_df: Latest valid record per NCT number (deduplicated curated silver table)
print("Curated :", silver_curated_df.count())

# Distinct NCT numbers in valid_df
print(
    "Distinct NCT :",
    valid_df.select(
        "nct_number"
    ).distinct().count()
)

# Log validation summary
log_silver_validation(bronze_df.count(), valid_df.count(), rejected_df.count(), "Silver Layer Validation Summary")

##### Merge curated data (Incremental write operation)

In [0]:
# Get a reference to the curated clinical trial studies Delta table
silver_table = DeltaTable.forName(
    spark,
    "clinical_datalakehouse_modernization.silver.clinical_trial_studies_curated"
)

# Merge the latest curated records into the Delta table:
# - If a record with the same NCT number exists, update it with new data
# - If no matching record exists, insert the new record
(
    silver_table.alias("target")
    .merge(
        silver_curated_df.alias("source"),
        "target.nct_number = source.nct_number"
    )
    .whenMatchedUpdateAll()   # Update all columns for matching NCT numbers
    .whenNotMatchedInsertAll() # Insert new records for NCT numbers not already in the table
    .execute()
)

# Log merge operation
log_silver_merge(silver_curated_df, "clinical_datalakehouse_modernization.silver.clinical_trial_studies_curated", "merge")


##### Store rejected data

In [0]:
# Store rejected clinical trial records in the Delta table for auditing and review
rejected_df.write \
.format("delta") \
.mode("append") \
.saveAsTable(
    "clinical_datalakehouse_modernization.silver.clinical_trial_rejected_records"
)

# Log rejected records write
log_bronze_write(rejected_df, "clinical_datalakehouse_modernization.silver.clinical_trial_rejected_records", "append")

##### store duplicate records

In [0]:
duplicate_log_df.drop(
    "study_duration_days",
    "enrollment_bucket",
    "has_child",
    "has_adult",
    "has_older_adult",
    "first_posted_year",
    "first_posted_month",
    "status_category"
).write \
.format("delta") \
.mode("append") \
.saveAsTable(
    "clinical_datalakehouse_modernization.silver.clinical_trial_duplicate_log"
)

# Log duplicate records write
log_bronze_write(duplicate_log_df, "clinical_datalakehouse_modernization.silver.clinical_trial_duplicate_log", "append")

##### Displaying the records for validation

In [0]:
display(
    valid_df.select(
        "nct_number",
        "study_title",
        "study_status"
    ).limit(20)
)

# Log display analytics
log_gold_analytics("Valid Records Display", "Sample of validated records")

In [0]:
valid_df.printSchema()

# Log schema verification
log_silver_transformation(valid_df, "Schema Verification")

In [0]:
display(
    valid_df.limit(20)
)

# Log final sample display
log_gold_analytics("Final Valid Data Sample", "Complete record preview")

##### displaying the dervied columns

In [0]:
display(
    valid_df.select(
        "nct_number",
        "study_status",
        "status_category",
        "age",
        "has_child",
        "has_adult",
        "has_older_adult",
        "enrollment",
        "enrollment_bucket",
        "study_duration_days"
    ).limit(20)
)

##### Reading the curated tables

In [0]:
# reading the curated table 
silver_df = spark.table(
    "clinical_datalakehouse_modernization.silver.clinical_trial_studies_curated"
)

# checking the curated count
print("Curated Count :", silver_df.count())

In [0]:
# checking the curated schema
silver_df.printSchema()

In [0]:
# displaying the curated sample data 
display(
    silver_df.limit(20)
)

In [0]:
# checking the curated data using simple columns
display(
    silver_df.select(
        "nct_number",
        "study_title",
        "study_status",
        "enrollment"
    ).limit(20)
)

In [0]:
# checking the NCT Format
display(
    silver_df.select("nct_number")
    .limit(20)
)

In [0]:
# checking the null value counts
silver_df.select(
    count(when(col("nct_number").isNull(), True)).alias("nct_nulls"),
    count(when(col("study_title").isNull(), True)).alias("title_nulls"),
    count(when(col("sponsor").isNull(), True)).alias("sponsor_nulls"),
    count(when(col("enrollment").isNull(), True)).alias("enrollment_nulls")
).display()

In [0]:
# Verify no nulls in study_duration_days from valid_df
valid_df.select(
    count(when(col("study_duration_days").isNull(), True)).alias("duration_nulls"),
    count(when(col("study_duration_days") == 0, True)).alias("duration_zeros")
).display()

In [0]:
# checking the derived columns
display(
    silver_df.select(
        "has_child",
        "has_adult",
        "has_older_adult",
        "first_posted_year",
        "first_posted_month"
    ).limit(20)
)


In [0]:
%sql
SELECT
  funder_type,
  COUNT(*) as count
FROM
  clinical_datalakehouse_modernization.silver.clinical_trial_studies_curated
GROUP BY
  funder_type
ORDER BY
  count DESC


In [0]:
%sql
SELECT
  funder_type,
  COUNT(*) as count
FROM
  clinical_datalakehouse_modernization.silver.clinical_trial_studies_curated
GROUP BY
  funder_type
ORDER BY
  count DESC


In [0]:
%sql
SELECT
  phases,
  COUNT(*) as count
FROM
  clinical_datalakehouse_modernization.silver.clinical_trial_studies_curated
GROUP BY
  phases
ORDER BY
  count DESC
